# Preprocesamiento de Vida O Muerte: Catálogo Médico para Machine Learning y Deep Learning

Cuando exploramos los increíbles datasets clínicos de `0.data.md` (Como el Breast Cancer Wisconsin o los diagnósticos pulmonares a través de rayos X), descubrimos un patrón sombrío: La información médica en el mundo real está repleta de ruidos y valores faltantes.

Un falso negativo porque a un modelo le hizo ruido un espacio vacío en el conteo de leucocitos, puede valer vidas. En esta guía interactiva veremos cómo blindamos los Datos Tabulares y las Imágenes Biológicas en la práctica médica profesional.

---

## PARTE 1: La Enfermería de Datos (Datos Estructurados o Tabulares)

Datasets como el **Medical Costs Personal Dataset** y el **Heart Disease Health Indicators** confían ciegamente en columnas clásicas de información sensoria y demográfica.

### 1.1 Rescatando Valores de Sangre Nulos (Imputación Avanzada)

Imagina que estamos prediciendo si un tumor de seno es Benigno o Maligno, y se perdió la medida de "Radio y Perímetro Nuclear" en el laboratorio.
Borrar al paciente altera brutalmente nuestra escaza muestra másica. Si ponemos un promedio global, alteramos el caso.
Para medicina severa usamos algoritmos biomiméticos como `KNNImputer`: La máquina observará qué pacientes tenían tumores casi idénticos en su patrón e inferirá ese número nuclear perdido fijándose en sus "Vecinos genéticos Cercanos".

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer

# Exámenes pélvicomamarios de la paciente 1 a la 3
df_tumores = pd.DataFrame({
    'Textura_Celular': [10.3, 15.6, 10.5],
    'Area_Tumoral': [1000, 2500, np.nan] # A la paciente 3 se le perdió el valor de su área
})

print("Datos Crudos de Tejido:")
display(df_tumores)

# KNN encontrará a la señora con la "Textura_Celular" más parecida al paciente 3, y asumirá un Area_Tumoral coherente con ese par.
imputador_vecinos = KNNImputer(n_neighbors=1)
df_salvado = pd.DataFrame(imputador_vecinos.fit_transform(df_tumores), columns=['Textura_Celular', 'Area_Tumoral_Reparada'])
print("\nDatos reconstruidos por simulitud epigenética:")
display(df_salvado)

### 1.2 Codificando Tipo de Sangre y Etapas del Cáncer (Encoding)

No es lo mismo catalogar algo nominal (Tener sangre A+, O+, AB) que algo Ordinal o progresivo (Cáncer Etapa I, Etapa II, Etapa IV metástasis).
- Si usas números para la sangre (1, 2, 3), la Fórmula Matemática deducirá erróneamente que Sangre AB (3) "vale el triple" que Sangre O (1). Usamos binarios purísimos `OneHotEncoder`.
- Para la metástasis oncológica, sí hay orden y gravedad matemática de peor a mejor. Usamos `OrdinalEncoder`.

In [ ]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

historiales = pd.DataFrame({
    'Sangre': ['O', 'A', 'O'], 
    'Etapa_Tumor': ['Fase I', 'Fase III', 'Recuperado']
})
display(historiales)

# Ordinal: El cáncer SÍ progresa en jerarquía negativa
ordinal = OrdinalEncoder(categories=[['Recuperado', 'Fase I', 'Fase III']])
historiales['Gravedad_Numerica'] = ordinal.fit_transform(historiales[['Etapa_Tumor']])

print("\nCómo la Red Neuronal procesa realmente tu gravedad clínica:", list(historiales['Gravedad_Numerica']))

### 1.3 Calibrando Señales Fisiológicas Extremas (Scaling MinMax vs Z-Score)

En un pre-ingreso a cuidados intensivos (MIMIC-III o Heart Disease), tendemos números tan dispares como una proteína en sangre que marca "0.003" mg y Cálculo de Aseguradora Médica marcando "85,000" USD.
Para evitar el aplastamiento algorítmico, utilizamos escalamientos severos robustos.

---

## PARTE 2: Visión Artificial (Radiología, Histología y Convoluciones)

Cuando abordamos los Datasets radiográficos como **Chest X-Ray (Neumonía)**, ya no tenemos CSVs. Tenemos máquinas antiguas o distintas que varían el tamaño, la resolución de los píxeles y el brillo (Resizing y Rescaling).

### 2.1 Rescaling Biológico y Resizing Exacto

Toda radiografía digital DICOM o Jpeg llega con resoluciones dispares (ej. HD y 4K mezclados). Y sus escalas de blancos y negros (Huesos a Aire denso) están entre 0 a 255. Redes biomédicas como ResNet implorán tamaños de matriz perfectos (224x224) y colores de escala decimal (0.00 negro absoluto a 1.00 luz absoluta de los huesos).

In [ ]:
import numpy as np

# Simulemos la mancha de aire y costilla de un Rx de Torax
rx_torax = np.array([
    [250, 40],   # Costilla hiperclara 
    [0,  150]    # Aire pulmonar (negro)
])

# Compresión de intensidades densométricas al nivel neuronal:
rx_torax_normalizado = rx_torax / 255.0
print("Matriz tensorial radiográfica preprocesada lista para convolución profunda:\n", np.round(rx_torax_normalizado, 3))

### 2.2 Rotando Núcleos Cancerosos en el Microspio (Data Augmentation)

Si tienes pocas imágenes microscópicas confirmadas de cáncer cerebral, el modelo las "memorizará" tontamente y pensará que sólo si ve la célula maligna «mirando a la derecha» es de riesgo (*Overfitting*).

Para romper su vicio y enseñarle las auténticas *texturas tumorísticas* indistintamente, volteamos, cortamos y escalamos cada muestra microscópica generando enormes bases de fotogramas médicos artificales validados.

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Preprocesador in-vitro de portaobjetos médicos
aumentador_celulas_cancerosas = ImageDataGenerator(
    rescale=1./255,           # Ajuste microscopico obligatorio a [0, 1]
    rotation_range=90,        # Una célula histológica NO cambia si se rota 90º (Biología)
    horizontal_flip=True,     # Los núcleos pueden presentarse en espejos visuales libres
    vertical_flip=True,       # Aumentando el dataset x4 veces sólo con volteos axiales
    zoom_range=0.10           # Simulando lentes de aumento óptico ligeramente variantes
)

print("Toda la IA oncológica usa esto antes de dar su primera mirada real a una biopsia en el quirófano.")